In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/CooperUnion/anime-recommendations-database/rating.csv
/kaggle/input/datasets/organizations/CooperUnion/anime-recommendations-database/anime.csv


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
rating = pd.read_csv("/kaggle/input/datasets/organizations/CooperUnion/anime-recommendations-database/rating.csv")

In [ ]:
anime = pd.read_csv("/kaggle/input/datasets/organizations/CooperUnion/anime-recommendations-database/anime.csv")

In [ ]:
rating['rating'] = rating['rating'].replace(-1, np.nan)
print(rating.head())

   user_id  anime_id  rating
0        1        20     NaN
1        1        24     NaN
2        1        79     NaN
3        1       226     NaN
4        1       241     NaN


In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler

print("🚀 BƯỚC 1: Đọc dữ liệu...")
# Đọc cả hai file, nhưng lúc này ta sẽ tập trung xử lý file anime trước 
# vì nó chứa các Feature (Đặc trưng) cần thiết cho Content-Based
anime_df = pd.read_csv('/kaggle/input/datasets/organizations/CooperUnion/anime-recommendations-database/anime.csv')
rating_df = pd.read_csv('/kaggle/input/datasets/organizations/CooperUnion/anime-recommendations-database/rating.csv') 

print("🧹 BƯỚC 2: Làm sạch dữ liệu...")
# Bỏ qua các dòng bị thiếu dữ liệu ở các cột quan trọng
anime_clean = anime_df.dropna(subset=['genre', 'rating', 'members']).copy()

# Xử lý cột episodes: Chuyển 'Unknown' thành NaN, sau đó có thể điền bằng 1 hoặc bỏ qua
anime_clean['episodes'] = pd.to_numeric(anime_clean['episodes'], errors='coerce')
anime_clean['episodes'] = anime_clean['episodes'].fillna(1)

print("⚙️ BƯỚC 3: Feature Engineering - One-Hot Encoding cho Thể loại...")
# Tách chuỗi thành List
anime_clean['genre_list'] = anime_clean['genre'].str.split(', ')

# Dùng MultiLabelBinarizer để tạo ma trận 0-1
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(anime_clean['genre_list'])

# Tạo DataFrame mới chứa các cột thể loại và ép kiểu int8 để siêu tiết kiệm RAM
genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_, index=anime_clean.index)
for col in genre_df.columns:
    genre_df[col] = genre_df[col].astype('int8')

# Nối ma trận thể loại vào bảng chính
anime_features = pd.concat([anime_clean, genre_df], axis=1)

print("⚖️ BƯỚC 4: Feature Scaling - Chuẩn hóa Điểm số và Lượt xem...")
# Khởi tạo công cụ MinMaxScaler (ép mọi con số về khoảng từ 0 đến 1)
scaler = MinMaxScaler()

# Chúng ta tạo ra 2 cột mới có hậu tố '_scaled' để giữ lại dữ liệu gốc nếu cần nhìn lại
anime_features[['rating_scaled', 'members_scaled']] = scaler.fit_transform(anime_features[['rating', 'members']])

print("✅ HOÀN TẤT! Dưới đây là diện mạo dữ liệu đã sẵn sàng cho Thuật toán:")
# In thử 5 dòng đầu tiên với các cột quan trọng để kiểm tra
columns_to_show = ['name', 'rating', 'rating_scaled', 'members', 'members_scaled', 'Action', 'Romance']
print(anime_features[columns_to_show].head())

🚀 BƯỚC 1: Đọc dữ liệu...
🧹 BƯỚC 2: Làm sạch dữ liệu...
⚙️ BƯỚC 3: Feature Engineering - One-Hot Encoding cho Thể loại...
⚖️ BƯỚC 4: Feature Scaling - Chuẩn hóa Điểm số và Lượt xem...
✅ HOÀN TẤT! Dưới đây là diện mạo dữ liệu đã sẵn sàng cho Thuật toán:
                               name  rating  rating_scaled  members  \
0                    Kimi no Na wa.    9.37       0.924370   200630   
1  Fullmetal Alchemist: Brotherhood    9.26       0.911164   793665   
2                          Gintama°    9.25       0.909964   114262   
3                       Steins;Gate    9.17       0.900360   673572   
4                     Gintama&#039;    9.16       0.899160   151266   

   members_scaled  Action  Romance  
0        0.197867       0        1  
1        0.782769       1        0  
2        0.112683       1        0  
3        0.664323       0        0  
4        0.149180       1        0  


In [ ]:
anime_features.head()

,anime_id,name,genre,type,episodes,rating,members,genre_list,Action,Adventure,...,Space,Sports,Super Power,Supernatural,Thriller,Vampire,Yaoi,Yuri,rating_scaled,members_scaled
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1.0,9.37,200630,"[Drama, Romance, School, Supernatural]",0,0,...,0,0,0,1,0,0,0,0,0.924370,0.197867
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64.0,9.26,793665,"[Action, Adventure, Drama, Fantasy, Magic, Mil...",1,1,...,0,0,0,0,0,0,0,0,0.911164,0.782769
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51.0,9.25,114262,"[Action, Comedy, Historical, Parody, Samurai, ...",1,0,...,0,0,0,0,0,0,0,0,0.909964,0.112683
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24.0,9.17,673572,"[Sci-Fi, Thriller]",0,0,...,0,0,0,0,1,0,0,0,0.900360,0.664323
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51.0,9.16,151266,"[Action, Comedy, Historical, Parody, Samurai, ...",1,0,...,0,0,0,0,0,0,0,0,0.899160,0.149180


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("🧠 1. Đang chuẩn bị ma trận đặc trưng...")
# Chúng ta phải loại bỏ các cột chữ (tên phim, thể loại text) vì máy tính chỉ tính toán được trên số
columns_to_drop = ['anime_id', 'name', 'genre', 'type', 'rating', 'members', 'genre_list']

# Kiểm tra xem các cột này có tồn tại không trước khi drop để tránh lỗi
cols_to_drop_actual = [col for col in columns_to_drop if col in anime_features.columns]
features_matrix = anime_features.drop(columns=cols_to_drop_actual)

print("⏳ 2. Đang tính toán Ma trận tương đồng (Quá trình này có thể mất vài chục giây)...")
# Tính toán độ tương đồng giữa TẤT CẢ các bộ phim với nhau
similarity_matrix = cosine_similarity(features_matrix)
print("✅ Tính toán xong!")

# ---------------------------------------------------------
# 3. HÀM GỢI Ý PHIM CHÍNH THỨC
# ---------------------------------------------------------
def recommend_anime_1(title, df, sim_matrix, top_n=5):
    """
    Hàm này nhận vào tên phim, tìm các phim giống nó nhất và in ra kết quả.
    """
    try:
        # Bước A: Tìm vị trí (index) của bộ phim trong bảng dữ liệu
        idx = df[df['name'] == title].index[0]
        
        # Bước B: Lấy điểm tương đồng của phim này với toàn bộ các phim khác
        # enumerate giúp giữ lại vị trí index ban đầu (index, score)
        sim_scores = list(enumerate(sim_matrix[idx]))
        
        # Bước C: Sắp xếp các phim theo điểm tương đồng từ cao xuống thấp
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        # Bước D: Lấy top N phim giống nhất (Bỏ qua phim đầu tiên ở vị trí [0] vì đó chính là nó)
        top_movies = sim_scores[1:top_n+1]
        
        # Bước E: Lấy tên phim từ index
        movie_indices = [i[0] for i in top_movies]
        
        print(f"🎬 Nếu bạn thích '{title}', bạn có thể sẽ thích:")
        for i, idx in enumerate(movie_indices, 1):
            movie_name = df['name'].iloc[idx]
            match_score = top_movies[i-1][1] * 100 # Đổi ra phần trăm
            print(f"  {i}. {movie_name} (Độ tương đồng: {match_score:.1f}%)")
            
    except IndexError:
        print(f"❌ Không tìm thấy phim '{title}' trong dữ liệu. Hãy gõ chính xác tên tiếng Anh/Romaji nhé!")

# ---------------------------------------------------------
# 4. CHẠY THỬ NGHIỆM
# ---------------------------------------------------------
print("\n" + "="*40)
# Thử tìm gợi ý cho một bộ phim kinh điển
recommend_anime_1('Naruto', anime_features, similarity_matrix)

print("-" * 40)
# Thử một bộ phim tình cảm
recommend_anime_1('Kimi no Na wa.', anime_features, similarity_matrix)

🧠 1. Đang chuẩn bị ma trận đặc trưng...
⏳ 2. Đang tính toán Ma trận tương đồng (Quá trình này có thể mất vài chục giây)...
✅ Tính toán xong!

🎬 Nếu bạn thích 'Naruto', bạn có thể sẽ thích:
  1. Katekyo Hitman Reborn! (Độ tương đồng: 100.0%)
  2. Dragon Ball Z (Độ tương đồng: 100.0%)
  3. Bleach (Độ tương đồng: 100.0%)
  4. Keroro Gunsou (Độ tương đồng: 100.0%)
  5. Kabatotto (Độ tương đồng: 100.0%)
----------------------------------------
🎬 Nếu bạn thích 'Kimi no Na wa.', bạn có thể sẽ thích:
  1. Aura: Maryuuin Kouga Saigo no Tatakai (Độ tương đồng: 91.5%)
  2. Clannad: After Story - Mou Hitotsu no Sekai, Kyou-hen (Độ tương đồng: 90.9%)
  3. Kokoro ga Sakebitagatterunda. (Độ tương đồng: 90.8%)
  4. Angel Beats!: Another Epilogue (Độ tương đồng: 90.7%)
  5. Harmonie (Độ tương đồng: 90.4%)


In [ ]:
recommend_anime_1("Nagi no Asukara", anime_features, similarity_matrix)

🎬 Nếu bạn thích 'Nagi no Asukara', bạn có thể sẽ thích:
  1. Winter Sonata (Độ tương đồng: 99.9%)
  2. Ningyohime Marina no Bouken (Độ tương đồng: 99.9%)
  3. Romeo x Juliet (Độ tương đồng: 99.9%)
  4. Kemono no Souja Erin (Độ tương đồng: 99.9%)
  5. Saiunkoku Monogatari (Độ tương đồng: 99.9%)


In [ ]:
# ==========================================
# 1. COLLABORATIVE FILTERING (Dựa trên Hành vi)
# ==========================================
print("⚙️ BƯỚC 1: Xử lý file rating.csv...")
# Lọc bỏ những dòng chưa chấm điểm (-1)
rating_clean = rating_df[rating_df['rating'] != -1]

# KỸ THUẬT LỌC NHIỄU VÀ BẢO VỆ RAM KAGGLE:
# Chỉ giữ lại những User đã xem > 50 phim và những Phim có > 50 người xem # bo cai nay
user_counts = rating_clean['user_id'].value_counts()
anime_counts = rating_clean['anime_id'].value_counts()


rating_filtered = rating_clean[
    (rating_clean['user_id'].isin(user_counts)) & 
    (rating_clean['anime_id'].isin(anime_counts))
]

print("⏳ BƯỚC 2: Tạo Pivot Table (Ma trận User-Anime)...")
# Ma trận khổng lồ: Hàng là Anime, Cột là User, Ô là Điểm
user_item_matrix = rating_filtered.pivot_table(index='anime_id', columns='user_id', values='rating').fillna(0)

print("🧠 BƯỚC 3: Tính Cosine Similarity cho Hành vi...")
collab_similarity = cosine_similarity(user_item_matrix)
collab_sim_df = pd.DataFrame(collab_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

print("✅ Đã có ma trận Collaborative (collab_sim_df)!")

⚙️ BƯỚC 1: Xử lý file rating.csv...
⏳ BƯỚC 2: Tạo Pivot Table (Ma trận User-Anime)...
🧠 BƯỚC 3: Tính Cosine Similarity cho Hành vi...
✅ Đã có ma trận Collaborative (collab_sim_df)!


In [ ]:
import optuna
from sklearn.metrics import mean_squared_error
import numpy as np
import joblib

print("⚙️ BƯỚC 4: Đồng bộ hóa 2 ma trận...")
content_sim_df = pd.DataFrame(similarity_matrix, index=features_matrix.index, columns=features_matrix.index)
common_anime_ids = list(set(content_sim_df.index) & set(collab_sim_df.index))

content_aligned = content_sim_df.loc[common_anime_ids, common_anime_ids].values
collab_aligned = collab_sim_df.loc[common_anime_ids, common_anime_ids].values

# Lọc luôn bảng User-Item gốc cho khớp với số lượng phim hiện tại
user_item_aligned = user_item_matrix.loc[common_anime_ids].values

# ---------------------------------------------------------
# HÀM ĐÁNH GIÁ (EVALUATE MODEL) BẰNG TOÁN HỌC
# ---------------------------------------------------------
import optuna
from sklearn.metrics import mean_squared_error
import numpy as np
import joblib
import pandas as pd

# (Giả định bạn đã chạy Bước 4 và có user_item_aligned, content_aligned, collab_aligned)

print("⚙️ BƯỚC 4.5: Chia tập Train/Test (Che giấu dữ liệu)...")
def split_train_test(user_matrix, test_ratio=0.2):
    """
    Hàm này sẽ "giấu" đi một lượng phần trăm (test_ratio) các ô đã có điểm chấm.
    """
    train_matrix = user_matrix.copy()
    test_matrix = np.zeros_like(user_matrix)
    
    # Tìm tọa độ (hàng, cột) của tất cả các ô có rating > 0
    nonzero_indices = np.argwhere(user_matrix > 0)
    
    # Chọn ngẫu nhiên 20% trong số đó để làm bài Test
    num_test = int(len(nonzero_indices) * test_ratio)
    test_indices_idx = np.random.choice(len(nonzero_indices), num_test, replace=False)
    test_indices = nonzero_indices[test_indices_idx]
    
    # Bốc các rating đó sang test_matrix và xóa (cho bằng 0) ở train_matrix
    for i, j in test_indices:
        test_matrix[i, j] = user_matrix[i, j]
        train_matrix[i, j] = 0
        
    return train_matrix, test_matrix

# Tạo 2 ma trận riêng biệt
train_matrix, test_matrix = split_train_test(user_item_aligned, test_ratio=0.2)

# ---------------------------------------------------------
# HÀM ĐÁNH GIÁ (EVALUATE MODEL) - ĐÃ FIX LỖI TOÁN HỌC
# ---------------------------------------------------------
def evaluate_model(hybrid_sim, train_mat, test_mat):
    """
    Dùng train_mat để dự đoán, và dùng test_mat để đối chiếu kết quả.
    """
    # Bước 1: Tử số (Độ tương đồng x Điểm user đã chấm trong tập Train)
    numerator = hybrid_sim.dot(train_mat)
    
    # Bước 2: Mẫu số - CHỈ tính tổng tương đồng của những phim user ĐÃ CHẤM (Fix lỗi quan trọng!)
    train_mask = (train_mat > 0).astype(float) # Chuyển đổi: phim nào đã xem là 1, chưa xem là 0
    denominator = np.abs(hybrid_sim).dot(train_mask) 
    
    denominator[denominator == 0] = 1e-9 # Tránh lỗi chia cho 0
    
    # Bước 3: Tính điểm dự đoán
    predicted_ratings = numerator / denominator
    
    # Bước 4: TÍNH RMSE CHỈ TRÊN NHỮNG Ô CỦA TẬP TEST (Những phim ta đã cố tình giấu đi)
    test_mask = test_mat > 0
    mse = mean_squared_error(test_mat[test_mask], predicted_ratings[test_mask])
    rmse = np.sqrt(mse)
    
    return rmse

# ---------------------------------------------------------
# HÀM CHO OPTUNA CHẠY
# ---------------------------------------------------------
def objective(trial):
    alpha = trial.suggest_float('alpha', 0.0, 1.0)
    
    hybrid_sim = (alpha * content_aligned) + ((1 - alpha) * collab_aligned)
    
    # Đưa train và test vào hàm evaluate
    error_score = evaluate_model(hybrid_sim, train_matrix, test_matrix)
    
    return error_score

print("🚀 BƯỚC 5: Kích hoạt Optuna tìm Tỷ lệ Vàng...")
study = optuna.create_study(direction='minimize')

# Chạy thử 50 kịch bản (Vì dữ liệu đã chia train/test nên chạy 50 vòng cho chắc)
study.optimize(objective, n_trials=50) 

best_alpha = study.best_params['alpha']
best_error = study.best_value

print("\n" + "="*50)
print(f"🔥 KẾT QUẢ TỪ OPTUNA:")
print(f" - Tỷ lệ Nội dung (Content): {best_alpha:.2f}")
print(f" - Tỷ lệ Cộng đồng (Collab): {(1 - best_alpha):.2f}")
print(f" - Mức sai số (RMSE) thấp nhất trên tập TEST: {best_error:.4f}")
print("="*50)

# ---------------------------------------------------------
# CÁC BƯỚC LƯU FILE GIỮ NGUYÊN NHƯ CỦA BẠN
# ---------------------------------------------------------

print("💾 BƯỚC 6: Tạo ma trận chốt và Lưu file...")
# Ráp ma trận cuối cùng bằng tỷ lệ hoàn hảo Optuna vừa tìm được
final_hybrid_matrix = (best_alpha * content_aligned) + ((1 - best_alpha) * collab_aligned)

# Gắn lại index và columns để lúc sau lên Web tìm theo ID cho dễ
final_hybrid_df = pd.DataFrame(final_hybrid_matrix, index=common_anime_ids, columns=common_anime_ids)

# Đóng gói mang về VS Code
joblib.dump(final_hybrid_df, 'hybrid_model.pkl')

# Đóng gói luôn một file anime dictionary (chứa ID và Tên phim) để web có cái mà hiển thị
anime_dict = anime_features.set_index('anime_id')['name'].to_dict()
joblib.dump(anime_dict, 'anime_dict.pkl')

print("✅ Đã xuất thành công 2 file .pkl. Bạn tải về máy tính được rồi nhé!")

[I 2026-04-30 01:02:51,945] A new study created in memory with name: no-name-bb1fbe22-f586-4666-88ab-1e73c3e42785
[I 2026-04-30 01:02:52,008] Trial 0 finished with value: 1.5763657916142793 and parameters: {'alpha': 0.4716900370682062}. Best is trial 0 with value: 1.5763657916142793.


⚙️ BƯỚC 4: Đồng bộ hóa 2 ma trận...
⚙️ BƯỚC 4.5: Chia tập Train/Test (Che giấu dữ liệu)...
🚀 BƯỚC 5: Kích hoạt Optuna tìm Tỷ lệ Vàng...


[I 2026-04-30 01:02:52,070] Trial 1 finished with value: 1.5872043323664276 and parameters: {'alpha': 0.8739344469857098}. Best is trial 0 with value: 1.5763657916142793.
[I 2026-04-30 01:02:52,128] Trial 2 finished with value: 1.569400046243769 and parameters: {'alpha': 0.3100746307104443}. Best is trial 2 with value: 1.569400046243769.
[I 2026-04-30 01:02:52,187] Trial 3 finished with value: 1.5729148236731307 and parameters: {'alpha': 0.3850145817009689}. Best is trial 2 with value: 1.569400046243769.
[I 2026-04-30 01:02:52,245] Trial 4 finished with value: 1.5789645667174457 and parameters: {'alpha': 0.5475436292375543}. Best is trial 2 with value: 1.569400046243769.
[I 2026-04-30 01:02:52,305] Trial 5 finished with value: 1.5626024006208705 and parameters: {'alpha': 0.1939413259088114}. Best is trial 5 with value: 1.5626024006208705.
[I 2026-04-30 01:02:52,361] Trial 6 finished with value: 1.5518733798667042 and parameters: {'alpha': 0.06267616475839888}. Best is trial 6 with valu


🔥 KẾT QUẢ TỪ OPTUNA:
 - Tỷ lệ Nội dung (Content): 0.00
 - Tỷ lệ Cộng đồng (Collab): 1.00
 - Mức sai số (RMSE) thấp nhất trên tập TEST: 1.5452
💾 BƯỚC 6: Tạo ma trận chốt và Lưu file...
✅ Đã xuất thành công 2 file .pkl. Bạn tải về máy tính được rồi nhé!


In [3]:
# thu lai lan hai 
# rating, anime la hai file csv
# su dung two tower architecture ket hop voi matrix factorrization
# chia thanh anime tower va user tower
from sklearn.preprocessing import MultiLabelBinarizer

# xu li rating 
rating_df = pd.read_csv('/kaggle/input/datasets/organizations/CooperUnion/anime-recommendations-database/rating.csv')
anime_df = pd.read_csv('/kaggle/input/datasets/organizations/CooperUnion/anime-recommendations-database/anime.csv')


NameError: name 'pd' is not defined

In [2]:
anime_df

NameError: name 'anime_df' is not defined